In [28]:
!pip -q install openai
!pip -q install gradio
!pip -q install tavily-python

In [29]:
import os
from getpass import getpass
from openai import OpenAI
from tavily import TavilyClient

In [30]:


os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.1-8b-instant"

Enter your Groq API Key: ··········


In [31]:
def run_agent(agent_name, prompt, temperature=0.2):
    """
    Runs one AI agent and returns its response.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": f"You are {agent_name}. Follow the user's instructions carefully."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature
    )

    return response.choices[0].message.content

In [32]:


os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API Key: ")

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def search_tool(query, max_results=5):
    """
    Live web search tool.
    Returns formatted search results for the agents.
    """
    if not query or not query.strip():
        return "Search error: query is empty."

    try:
        results = tavily_client.search(
            query=query,
            search_depth="basic",
            max_results=max_results
        )

        formatted_results = []

        for i, item in enumerate(results.get("results", []), 1):
            title = item.get("title", "No title")
            url = item.get("url", "No URL")
            content = item.get("content", "No content")

            formatted_results.append(
                f"{i}. Title: {title}\nURL: {url}\nContent: {content}"
            )

        if not formatted_results:
            return "No search results found."

        return "\n\n".join(formatted_results)

    except Exception as e:
        return f"Search tool error: {e}"

Enter your Tavily API Key: ··········


In [33]:
def calculator_tool(expression):
    """
    Simple calculator tool for finance calculations.
    """
    try:
        allowed_chars = "0123456789+-*/(). "
        if not all(char in allowed_chars for char in expression):
            return "Calculator error: invalid characters in expression."

        result = eval(expression)
        return result

    except Exception as e:
        return f"Calculator error: {e}"

In [41]:
import time

ALLOWED_SCENARIOS = ["Optimistic", "Baseline", "Crisis"]


def run_agent(agent_name, prompt, temperature=0.2):
    """
    Runs one AI agent and returns its response.
    Includes retry handling for Groq rate limit errors.
    """

    max_retries = 5

    if agent_name == "Business Plan Agent":
        max_tokens = 1200
    elif agent_name == "Validation Agent":
        max_tokens = 300
    else:
        max_tokens = 700

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": f"You are {agent_name}. Follow the user's instructions carefully."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )

            return response.choices[0].message.content

        except Exception as e:
            error_message = str(e)

            if "429" in error_message or "rate_limit" in error_message or "Rate limit" in error_message:
                wait_time = 10 + (attempt * 10)
                print(f"Rate limit reached. Waiting {wait_time} seconds before retrying...")
                time.sleep(wait_time)
            else:
                raise e

    return "Error: Rate limit reached. Please try again later."


def run_competition_agent():
    competition_query = f"""
real competitors in {latest_startup["City"]} for a {latest_startup["Business Type"]} business
startup description: {latest_startup["Project Idea / Description"]}
"""

    competition_search_results = search_tool(competition_query)

    competition_prompt = f"""
You are a Competition Analysis Agent.

Startup details:
Project Name: {latest_startup["Project Name"]}
City: {latest_startup["City"]}
Business Type: {latest_startup["Business Type"]}
Budget: {latest_startup["Capital / Budget"]} SAR
Description: {latest_startup["Project Idea / Description"]}

Market Agent Report:
{latest_reports["market"]}

Live Search Results:
{competition_search_results}

Important rules:
- Analyze competitors based only on the selected scenario(s) from the Market Agent.
- Do NOT choose a new scenario.
- Use only these scenarios if mentioned: Optimistic, Baseline, Crisis.
- Do NOT write Mixed.
- Use Live Search Results when mentioning competitor names.
- Do not invent competitor names.

Return:
- Selected Scenario(s)
- Direct Competitors
- Indirect Competitors
- Competitor Strengths
- Competitor Weaknesses
- Competitive Advantage Suggestions
- Scenario-Based Competition Recommendation

Keep it short.
"""

    competition_report = run_agent("Competition Analysis Agent", competition_prompt)

    latest_reports["competition"] = competition_report
    latest_reports["competition_search_results"] = competition_search_results

    return competition_report


def run_finance_agent():
    budget_text = str(latest_startup["Capital / Budget"]).replace(",", "").strip()
    budget = float(budget_text)
    business_type = latest_startup["Business Type"]

    if business_type == "App":
        development_cost = calculator_tool(f"{budget} * 0.40")
        hosting_tools_cost = calculator_tool(f"{budget} * 0.15")
        marketing_cost = calculator_tool(f"{budget} * 0.25")
        operations_cost = calculator_tool(f"{budget} * 0.10")
        reserve_cost = calculator_tool(f"{budget} * 0.10")

        allocation_total = (
            development_cost
            + hosting_tools_cost
            + marketing_cost
            + operations_cost
            + reserve_cost
        )

        finance_calculations = f"""
Original Budget from Form: {budget} SAR

Budget Allocation:
- App Development / MVP: 40% = {development_cost} SAR
- Hosting / APIs / Tools: 15% = {hosting_tools_cost} SAR
- Marketing / User Acquisition: 25% = {marketing_cost} SAR
- Operations / Support: 10% = {operations_cost} SAR
- Reserve: 10% = {reserve_cost} SAR
- Total Allocation: {allocation_total} SAR

Estimates:
- Estimate monthly costs based on hosting, tools, maintenance, support, and marketing.
- Estimate revenue based on subscriptions, premium features, ads, or service fees.
- Break-even should be estimated realistically for an app business.
"""

    elif business_type == "Store":
        inventory_cost = calculator_tool(f"{budget} * 0.45")
        setup_cost = calculator_tool(f"{budget} * 0.20")
        marketing_cost = calculator_tool(f"{budget} * 0.15")
        operations_cost = calculator_tool(f"{budget} * 0.10")
        reserve_cost = calculator_tool(f"{budget} * 0.10")

        allocation_total = (
            inventory_cost
            + setup_cost
            + marketing_cost
            + operations_cost
            + reserve_cost
        )

        finance_calculations = f"""
Original Budget from Form: {budget} SAR

Budget Allocation:
- Initial Inventory: 45% = {inventory_cost} SAR
- Store Setup / E-commerce Setup: 20% = {setup_cost} SAR
- Marketing: 15% = {marketing_cost} SAR
- Operations: 10% = {operations_cost} SAR
- Reserve: 10% = {reserve_cost} SAR
- Total Allocation: {allocation_total} SAR

Estimates:
- Estimate monthly costs based on inventory, rent or e-commerce tools, delivery, staff, and marketing.
- Estimate revenue based on product sales, online orders, bundles, and repeat customers.
- Break-even should be estimated realistically for a store business.
"""

    elif business_type == "Company":
        tools_cost = calculator_tool(f"{budget} * 0.25")
        hiring_cost = calculator_tool(f"{budget} * 0.30")
        marketing_cost = calculator_tool(f"{budget} * 0.20")
        operations_cost = calculator_tool(f"{budget} * 0.15")
        reserve_cost = calculator_tool(f"{budget} * 0.10")

        allocation_total = (
            tools_cost
            + hiring_cost
            + marketing_cost
            + operations_cost
            + reserve_cost
        )

        finance_calculations = f"""
Original Budget from Form: {budget} SAR

Budget Allocation:
- Tools / Software: 25% = {tools_cost} SAR
- Hiring / Freelancers: 30% = {hiring_cost} SAR
- Marketing / Sales: 20% = {marketing_cost} SAR
- Operations: 15% = {operations_cost} SAR
- Reserve: 10% = {reserve_cost} SAR
- Total Allocation: {allocation_total} SAR

Estimates:
- Estimate monthly costs based on tools, freelancers, operations, sales, and support.
- Estimate revenue based on service packages, client projects, monthly retainers, consulting, or support contracts.
- Break-even should be estimated realistically for a service company.
"""

    else:
        setup_cost = calculator_tool(f"{budget} * 0.45")
        equipment_cost = calculator_tool(f"{budget} * 0.25")
        marketing_cost = calculator_tool(f"{budget} * 0.15")
        reserve_cost = calculator_tool(f"{budget} * 0.15")

        allocation_total = setup_cost + equipment_cost + marketing_cost + reserve_cost

        finance_calculations = f"""
Original Budget from Form: {budget} SAR

Budget Allocation:
- Setup / Rent / Interior: 45% = {setup_cost} SAR
- Equipment: 25% = {equipment_cost} SAR
- Marketing: 15% = {marketing_cost} SAR
- Reserve: 15% = {reserve_cost} SAR
- Total Allocation: {allocation_total} SAR

Estimates:
- Estimate monthly costs based on rent, staff, supplies, utilities, and marketing.
- Estimate revenue based on sales, customer traffic, delivery, bundles, or subscriptions if suitable.
- Break-even should be estimated realistically for the selected business type.
"""

    finance_prompt = f"""
You are a Finance Planning Agent.

Startup:
Project Name: {latest_startup["Project Name"]}
City: {latest_startup["City"]}
Business Type: {latest_startup["Business Type"]}
Budget: {budget} SAR
Description: {latest_startup["Project Idea / Description"]}

Market Agent Report:
{latest_reports["market"]}

Competition Report:
{latest_reports["competition"]}

Calculator Tool Results:
{finance_calculations}

Important rules:
- Do NOT choose a new scenario.
- Copy the selected scenario(s) from the Market Agent Report.
- Use SAR only.
- Use the original budget from the form only.
- Do not invent a different budget.
- Do NOT write Mixed.
- Build the financial plan only for the selected scenario(s).
- Make the budget allocation suitable for the selected Business Type.
- Do not use rent/interior costs for App businesses unless the description clearly requires a physical office.
- Monthly cost, revenue estimate, and break-even estimate must match the Business Type.
- Keep the financial plan realistic and avoid exaggerated revenue assumptions.

Return:
* Selected Scenario(s)
* Budget Allocation
* Monthly Cost
* Revenue Estimate
* Break-even Estimate
* Financial Risks
* Recommendation

Keep it short.
"""

    finance_report = run_agent("Finance Planning Agent", finance_prompt)

    latest_reports["finance"] = finance_report
    latest_reports["finance_calculations"] = finance_calculations

    return finance_report


def run_risk_agent():
    risk_prompt = f"""
You are a Risk Analysis Agent.

Startup details:
Project Name: {latest_startup["Project Name"]}
City: {latest_startup["City"]}
Business Type: {latest_startup["Business Type"]}
Budget: {latest_startup["Capital / Budget"]} SAR
Description: {latest_startup["Project Idea / Description"]}

Market Report:
{latest_reports["market"]}

Competition Report:
{latest_reports["competition"]}

Finance Report:
{latest_reports["finance"]}

Important rules:
- Analyze risks based only on the selected scenario(s) from the Market Agent.
- Do NOT choose a new scenario.
- Use only: Optimistic, Baseline, Crisis.
- Do NOT write Mixed.
- Do not repeat previous reports.
- Make risks suitable for the selected Business Type.

Return:
- Selected Scenario(s)
- Main Risks
- Risk Level: Low / Medium / High
- Risk Impact
- Mitigation Plan
- Scenario-Based Risk Recommendation

Keep it short, maximum 6 bullet points.
"""

    risk_report = run_agent("Risk Analysis Agent", risk_prompt)

    latest_reports["risk"] = risk_report

    return risk_report


def run_marketing_agent():
    marketing_prompt = f"""
You are a Marketing Strategy Agent.

Startup:
Project Name: {latest_startup["Project Name"]}
City: {latest_startup["City"]}
Business Type: {latest_startup["Business Type"]}
Budget: {latest_startup["Capital / Budget"]} SAR
Description: {latest_startup["Project Idea / Description"]}

Market Summary:
{latest_reports["market"]}

Risk Summary:
{latest_reports["risk"]}

Important rules:
- Create marketing strategy based only on the selected scenario(s).
- Do NOT choose a new scenario.
- Use only: Optimistic, Baseline, Crisis.
- Do NOT write Mixed.
- Do not repeat previous reports.
- Make marketing ideas suitable for the selected Business Type.
- Keep marketing actions realistic for the budget.

Return only:
- Selected Scenario(s)
- Target Audience
- Marketing Channels
- Promotion Ideas
- Marketing Recommendation

Keep it short.
"""

    marketing_report = run_agent("Marketing Strategy Agent", marketing_prompt)

    latest_reports["marketing"] = marketing_report

    return marketing_report


def run_business_plan_agent():
    budget_text = str(latest_startup["Capital / Budget"]).replace(",", "").strip()
    budget = float(budget_text)

    business_plan_prompt = f"""
You are a Business Plan Agent.

Startup:
Project Name: {latest_startup["Project Name"]}
City: {latest_startup["City"]}
Business Type: {latest_startup["Business Type"]}
Budget: {budget} SAR
Description: {latest_startup["Project Idea / Description"]}

Market:
{latest_reports["market"]}

Finance:
{latest_reports["finance"]}

Finance Calculations:
{latest_reports["finance_calculations"]}

Risk:
{latest_reports["risk"]}

Marketing:
{latest_reports["marketing"]}

Important rules:
- Do NOT create a new scenario.
- Copy the selected scenario(s) from the latest Market Report exactly.
- Use only: Optimistic, Baseline, Crisis.
- Do NOT write Mixed.
- Use SAR only.
- Use the original budget from the form only.
- The total budget allocation must equal exactly {budget} SAR.
- Do not write inconsistent numbers.
- Do not create a new budget allocation in the Financial Recommendation section.
- Any financial recommendation must follow the same budget allocation already shown in Section 2.
- Any marketing recommendation must stay within the marketing budget shown in Section 2.
- Revenue Breakdown must include realistic SAR amounts, not only percentages.
- Do not list free services as revenue sources.
- Do not list loyalty programs as direct revenue sources unless they have paid membership fees.
- Do not include vague revenue sources such as "Other Revenue" or "miscellaneous revenue" unless clearly explained with a realistic SAR amount.
- Make pricing realistic for the selected Business Type, city, target customers, and budget.
- Make the plan practical and specific, not general.
- The final recommendation should summarize the best next step without changing the budget allocation.
- The Budget Allocation section must copy the exact categories, percentages, and SAR amounts from Finance Calculations.
- Do not rename, remove, merge, or replace budget categories from Finance Calculations.
- Do not create a new budget allocation even if the user feedback asks for improvement.
- If the plan is revised after Not Approved, keep the same budget allocation from Finance Calculations.
- Financial Recommendation must explain how to use the existing allocation, not create a new one.
You MUST include these sections:

Project Name:
Selected Scenario(s):
Scenario Reason:

Final Business Plan:

1. Main Strategy:
Write a practical strategy specific to the startup idea, selected city, target customers, and business type.

2. Budget Allocation:
Copy the exact Budget Allocation from Finance Calculations.
Use the same category names, percentages, and SAR amounts.
Do not add new categories.
Do not remove categories.
Do not change the percentages.
The total must equal the original budget exactly.

3. Products, Services, or Packages:
Create 3 realistic offerings based on the selected Business Type.

Use the correct format:
- If the business is a Cafe or Restaurant: suggest menu items, bundles, customer packages, delivery options, or catering if suitable.
- If the business is a Store: suggest product categories, price ranges, bundles, and best-selling items.
- If the business is an App: suggest subscription plans, premium features, team plans, or in-app revenue options.
- If the business is a Company: suggest service packages, project-based pricing, consulting, or monthly support contracts.
- Do not include study room pricing unless the startup idea is related to studying, coworking, or room booking.
- Do not include custom development services for an App unless the app is clearly a B2B platform or software provider.

4. Pricing Strategy:
Create a realistic pricing strategy based on the startup idea, business type, city, and budget.

Include:
- Price range in SAR
- Reason for the pricing
- How the pricing matches the target customers
- Keep prices realistic and not exaggerated.

5. Marketing Strategy:
Give specific Instagram and TikTok campaign ideas.
Do not write general marketing advice.

Include at least:
- 2 Instagram ideas
- 2 TikTok ideas
- 1 partnership idea suitable for the business type

Rules:
- Marketing ideas must be practical for Riyadh.
- If mentioning discounts, campaigns, or influencer marketing, keep them realistic for the marketing budget.
- Do not recommend marketing spending that exceeds the marketing budget in Section 2.

6. Revenue Breakdown:
Create a realistic monthly revenue breakdown based on the selected Business Type and startup description.

Rules:
- Include SAR amounts for each revenue source.
- Include a simple calculation, such as number of customers × price.
- Do not use percentages only.
- Do not include free services as revenue.
- Do not include loyalty programs as direct revenue unless they are paid.
- Do not include vague items like "Other Revenue" unless explained clearly with a SAR amount.
- Include a Total Monthly Revenue Estimate.

Use suitable revenue sources only:
- For Cafe or Restaurant: food, drinks, delivery, bundles, catering, or subscriptions if suitable.
- For Store: product sales, online orders, bundles, seasonal items, or paid consultations if suitable.
- For App: subscriptions, premium features, ads, team plans, or service fees.
- For Company: service packages, client projects, monthly retainers, consulting, or support contracts.
- Do not include irrelevant revenue sources.

7. Risk Mitigation:
Give practical risks and how to reduce them.
The risks must match the selected Business Type and startup description.

8. Differentiation:
Explain clearly how this startup stands out from existing competitors.
The differentiation must be specific, not generic.

9. Financial Recommendation:
Give a realistic financial recommendation based on the original budget and the budget allocation already shown in Section 2.

Rules:
- Do not create a new budget allocation.
- Do not recommend percentages that conflict with Section 2.
- Mention how the startup should use the existing budget wisely.
- If suggesting marketing actions, keep them within the existing marketing budget.
- Keep the recommendation specific to the selected Business Type.

Final Recommendation:
One clear final recommendation.
Do not change the budget allocation here.

Keep it structured, clear, and specific.
"""

    business_plan_report = run_agent("Business Plan Agent", business_plan_prompt)

    latest_reports["business_plan"] = business_plan_report

    return business_plan_report


def run_validation_agent():
    budget_text = str(latest_startup["Capital / Budget"]).replace(",", "").strip()
    budget = float(budget_text)

    validation_prompt = f"""
You are a Validation Agent.

Review this final business plan:
{latest_reports["business_plan"]}

Original Budget from Form:
{budget} SAR

Finance Calculations:
{latest_reports["finance_calculations"]}

Check only:
1. Is the selected scenario clear?
2. Does the plan use only: Optimistic, Baseline, Crisis?
3. Did the plan avoid using Mixed as a scenario?
4. Is the plan realistic?
5. Is the plan not generic?
6. If one scenario was selected, is there only one plan?
7. If two scenarios were selected, are there only two plans?
8. Does the budget allocation match the original budget from the form?
9. Is the budget allocation suitable for the selected Business Type?
10. Does the Revenue Breakdown include SAR amounts and realistic calculations?
11. Did the plan avoid listing free services as revenue?
12. Did the plan avoid listing loyalty programs as direct revenue unless they are paid?
13. Does the Financial Recommendation avoid creating a new budget allocation?
14. Does the Financial Recommendation avoid percentages that conflict with Section 2?

Important rules:
- Do not invent a different expected budget.
- Validate against the original budget from the form only.
- If Status is Approved, Issues must be: None.
- If there are real issues, Status must be: Not Approved.
- Do not rewrite the business plan.

Return only:
Status: Approved or Not Approved
Issues: max 2 short bullets
Recommendation: one short sentence

Keep it very short.
"""

    validation_report = run_agent("Validation Agent", validation_prompt)

    latest_reports["validation"] = validation_report

    return validation_report


def run_full_workflow():
    """
    Runs all agents after the current Market Report.
    Adds short pauses to reduce rate limit errors.
    """

    run_competition_agent()
    time.sleep(5)

    run_finance_agent()
    time.sleep(5)

    run_risk_agent()
    time.sleep(5)

    run_marketing_agent()
    time.sleep(5)

    run_business_plan_agent()
    time.sleep(5)

    run_validation_agent()

    return latest_reports["business_plan"], latest_reports["validation"]


def run_market_revision_then_full_workflow(additional_information):
    """
    Runs Market Revision, then runs the full workflow again.
    """

    revision_count = latest_reports.get("revision_count", 0) + 1
    latest_reports["revision_count"] = revision_count

    market_revision_prompt = f"""
You are a Market Revision Agent.

Re-analyze the startup idea after human feedback.

Original startup details:
Project Name: {latest_startup["Project Name"]}
City: {latest_startup["City"]}
Business Type: {latest_startup["Business Type"]}
Budget: {latest_startup["Capital / Budget"]} SAR
Description: {latest_startup["Project Idea / Description"]}

Current Market Report:
{latest_reports["market"]}

Human feedback:
{additional_information}

Allowed scenarios:
- Optimistic
- Baseline
- Crisis

Important rules:
- Do NOT write Mixed.
- Choose ONE scenario only, or TWO if the situation is between two scenarios.
- If the situation is mixed, write the two exact scenarios, such as Baseline and Optimistic.
- Do not create all three scenarios unless all three truly apply.
- Tell the next agents to build the analysis only for the selected scenario(s).
- Human feedback should improve the plan content only.
- Do not recommend changing the original budget allocation.
- Do not suggest new budget percentages.
- The next agents must keep the Finance Agent budget allocation unchanged.
- If the user asks for better financial recommendations, improve the explanation only, not the allocation.
- Do not change the selected scenario unless the human feedback explicitly asks to change it.
- Do not change the original target customers unless the human feedback explicitly asks to change them.
- Improve the plan based on feedback while keeping the original startup description, business type, budget allocation, and target customers consistent.
Return:

Market Summary:
Target Customers:
Opportunities:
Challenges:
Selected Scenario:
Reason for Scenario Choice:
Recommendation for Next Agents:

Keep it short.
"""

    revised_market_report = run_agent("Market Revision Agent", market_revision_prompt)

    latest_reports["original_market"] = latest_reports.get("market", "")
    latest_reports["market"] = revised_market_report
    latest_reports["market_revision"] = revised_market_report

    final_plan, validation = run_full_workflow()

    return revised_market_report, final_plan, validation

In [42]:
import gradio as gr

if "latest_startup" not in globals():
    latest_startup = {}

if "latest_reports" not in globals():
    latest_reports = {}


website_css = """
body {
    background:
        radial-gradient(circle at 12% 12%, rgba(99, 102, 241, 0.18), transparent 30%),
        radial-gradient(circle at 88% 10%, rgba(20, 184, 166, 0.14), transparent 28%),
        linear-gradient(135deg, #f8fafc 0%, #ffffff 55%, #f1f5f9 100%);
}

.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
}

.hero {
    padding: 44px;
    border-radius: 34px;
    background: linear-gradient(135deg, #0f172a 0%, #111827 55%, #1e1b4b 100%);
    color: white;
    box-shadow: 0 28px 80px rgba(15, 23, 42, 0.28);
    margin-bottom: 24px;
}

.badge {
    display: inline-block;
    padding: 8px 15px;
    border-radius: 999px;
    background: rgba(255,255,255,0.12);
    border: 1px solid rgba(255,255,255,0.18);
    color: #f8fafc !important;
    font-size: 13px;
    margin-bottom: 18px;
}

.hero-title {
    font-size: 52px;
    font-weight: 900;
    letter-spacing: -1.8px;
    margin: 0;
    line-height: 1.1;
    color: #ffffff !important;
    text-shadow: 0 4px 18px rgba(0, 0, 0, 0.30);
}

.hero-text {
    font-size: 18px;
    color: #e5e7eb !important;
    line-height: 1.75;
    max-width: 780px;
    margin-top: 14px;
}

.form-card, .plan-card, .review-card {
    padding: 30px;
    border-radius: 30px;
    background: rgba(255, 255, 255, 0.96);
    border: 1px solid #e5e7eb;
    box-shadow: 0 20px 55px rgba(15, 23, 42, 0.09);
}

.side-card {
    padding: 32px;
    border-radius: 30px;
    background: linear-gradient(145deg, #0f172a 0%, #111827 55%, #1e1b4b 100%);
    color: white;
    box-shadow: 0 22px 60px rgba(15, 23, 42, 0.22);
    min-height: 500px;
}

.side-card h2 {
    color: white !important;
    font-size: 30px;
}

.side-card p {
    color: #d1d5db !important;
    line-height: 1.75;
}

.mini-card {
    padding: 18px;
    border-radius: 20px;
    background: rgba(255, 255, 255, 0.12);
    border: 1px solid rgba(255, 255, 255, 0.22);
    margin-bottom: 14px;
}

.mini-card b {
    color: #ffffff !important;
    font-size: 16px;
}

.mini-card span {
    color: #dbeafe !important;
    font-size: 14px;
    line-height: 1.65;
}

.plan-display {
    padding: 26px;
    border-radius: 24px;
    background: #ffffff;
    border: 1px solid #e5e7eb;
    box-shadow: 0 14px 40px rgba(15, 23, 42, 0.08);
    min-height: 520px;
}

.success-box {
    padding: 16px 18px;
    border-radius: 18px;
    background: #ecfdf5;
    border: 1px solid #a7f3d0;
    color: #065f46;
    font-weight: 700;
    line-height: 1.7;
}

.warning-box {
    padding: 16px 18px;
    border-radius: 18px;
    background: #fff7ed;
    border: 1px solid #fed7aa;
    color: #9a3412;
    font-weight: 700;
    line-height: 1.7;
}

.info-box {
    padding: 16px 18px;
    border-radius: 18px;
    background: #eff6ff;
    border: 1px solid #bfdbfe;
    color: #1e40af;
    font-weight: 700;
    line-height: 1.7;
}

.loading-box {
    padding: 16px 18px;
    border-radius: 18px;
    background: #f5f3ff;
    border: 1px solid #c4b5fd;
    color: #5b21b6;
    font-weight: 800;
    line-height: 1.7;
    animation: pulse 1.3s infinite;
}

@keyframes pulse {
    0% { opacity: 0.65; }
    50% { opacity: 1; }
    100% { opacity: 0.65; }
}

.single-line-input textarea {
    resize: none !important;
    overflow-y: hidden !important;
    min-height: 48px !important;
    max-height: 48px !important;
}

.single-line-input textarea::-webkit-scrollbar {
    display: none !important;
}

button {
    border-radius: 16px !important;
    font-weight: 800 !important;
    padding: 12px 18px !important;
}

textarea, input, select {
    border-radius: 15px !important;
}

footer {
    display: none !important;
}
"""


def start_and_show_plan(project_name, city, budget, business_type, description):
    global latest_startup, latest_reports

    if not project_name or not city or not budget or not business_type or not description:
        message = """
<div class="warning-box">
⚠️ Please complete all startup details before continuing.
</div>
"""
        yield (
            gr.update(visible=True),
            gr.update(visible=False),
            message,
            gr.update(value=""),
            gr.update(value=""),
            gr.update(value=""),
            gr.update(interactive=True)
        )
        return

    loading_message = """
<div class="loading-box">
⏳ FounderAI is generating your business plan... Please wait.
<br>
Market, competition, finance, risk, marketing, and business plan agents are working now.
</div>
"""

    yield (
        gr.update(visible=True),
        gr.update(visible=False),
        loading_message,
        gr.update(value=""),
        gr.update(value=""),
        gr.update(value=""),
        gr.update(interactive=False)
    )

    latest_startup = {
        "Project Name": project_name,
        "City": city,
        "Capital / Budget": budget,
        "Business Type": business_type,
        "Project Idea / Description": description
    }

    latest_reports.clear()
    latest_reports["revision_count"] = 0

    market_prompt = f"""
You are a Market Analysis Agent.

Analyze the startup idea and decide which market scenario fits it best.

Startup details:
Project Name: {latest_startup["Project Name"]}
City: {latest_startup["City"]}
Business Type: {latest_startup["Business Type"]}
Budget: {latest_startup["Capital / Budget"]} SAR
Description: {latest_startup["Project Idea / Description"]}

Analyze:
1. Market demand
2. Target customers
3. Market opportunities
4. Market challenges
5. How strong or weak the startup position is in this market

Allowed scenarios:
- Optimistic
- Baseline
- Crisis

Important rules:
- Choose ONE scenario only, or TWO if the situation is between two scenarios.
- Do NOT write Mixed.
- If the situation is mixed, write the two exact scenarios, such as Baseline and Optimistic.
- Tell the next agents to build the analysis only for the selected scenario or scenarios.

Return a short structured report:

Market Summary:
Target Customers:
Opportunities:
Challenges:
Selected Scenario:
Reason for Scenario Choice:
Recommendation for Next Agents:

Keep it short.
"""

    market_report = run_agent("Market Analysis Agent", market_prompt)
    latest_reports["market"] = market_report

    final_plan, validation = run_full_workflow()

    success_message = f"""
<div class="success-box">
✨ FounderAI generated a business plan for <b>{project_name}</b>. Review it below.
</div>
"""

    yield (
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(value=""),
        success_message,
        gr.update(value=final_plan),
        gr.update(value=""),
        gr.update(interactive=True)
    )


def review_plan(approval, additional_information):
    global latest_reports

    if approval == "Approved":
        latest_reports["human_approval"] = "Approved"

        message = """
<div class="success-box">
✅ Approved. The final business plan has been accepted.
</div>
"""
        return (
            message,
            gr.update(value=latest_reports["business_plan"]),
            gr.update(value="")
        )

    if approval == "Not Approved" and not additional_information.strip():
        latest_reports["human_approval"] = "Not Approved"

        message = """
<div class="warning-box">
⚠️ Please write what needs to be improved before submitting Not Approved.
</div>
"""
        return (
            message,
            gr.update(value=latest_reports["business_plan"]),
            gr.update()
        )

    latest_reports["human_approval"] = "Not Approved"
    latest_reports["additional_information"] = additional_information

    revised_market, final_plan, validation = run_market_revision_then_full_workflow(additional_information)

    latest_reports["last_revised_market"] = revised_market
    latest_reports["last_validation"] = validation

    message = f"""
<div class="info-box">
🔄 Revision #{latest_reports["revision_count"]} completed. The business plan has been updated.
</div>
"""

    return (
        message,
        gr.update(value=final_plan),
        gr.update(value="")
    )


def back_to_form():
    return (
        gr.update(visible=True),
        gr.update(visible=False)
    )


try:
    website_demo.close()
except:
    pass


with gr.Blocks(css=website_css, theme=gr.themes.Soft()) as website_demo:

    with gr.Group(visible=True) as form_page:
        gr.HTML("""
        <div class="hero">
            <div class="badge">FounderAI • Multi-Agent Startup Advisor</div>
            <h1 class="hero-title">Turn your startup idea into a business plan.</h1>
            <p class="hero-text">
                FounderAI uses specialized AI agents to analyze your market, competitors, finance,
                risks, marketing strategy, and final business direction.
            </p>
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=6):
                with gr.Group(elem_classes="form-card"):
                    gr.Markdown("## Startup Information")
                    gr.Markdown("Enter your project details, then FounderAI will generate your final business plan.")

                    project_name = gr.Textbox(
                        label="Project Name",
                        placeholder="Example: StudySip Cafe",
                        lines=1,
                        max_lines=1,
                        elem_classes="single-line-input"
                    )

                    with gr.Row():
                        city = gr.Textbox(
                            label="City",
                            placeholder="Example: Riyadh",
                            lines=1,
                            max_lines=1,
                            elem_classes="single-line-input"
                        )

                        budget = gr.Textbox(
                            label="Capital / Budget (SAR)",
                            placeholder="Example: 120000",
                            lines=1,
                            max_lines=1,
                            elem_classes="single-line-input"
                        )

                    business_type = gr.Dropdown(
                        choices=["Cafe", "Restaurant", "Store", "App", "Company"],
                        label="Business Type",
                        value="Cafe"
                    )

                    description = gr.Textbox(
                        label="Project Idea / Description",
                        placeholder="Describe your startup idea, target customers, and unique features...",
                        lines=6
                    )

                    start_button = gr.Button("Start & Generate Business Plan", variant="primary")
                    form_message = gr.HTML()

            with gr.Column(scale=4):
                gr.HTML('''
                <div class="side-card">
                    <h2>How FounderAI Works</h2>
                    <p>
                        Your idea goes through a multi-agent workflow that turns raw details into a structured business plan.
                    </p>

                    <div class="mini-card">
                        <b>01 · Market Scenario</b><br>
                        <span>Identifies whether your idea fits an Optimistic, Baseline, or Crisis scenario.</span>
                    </div>

                    <div class="mini-card">
                        <b>02 · Business Agents</b><br>
                        <span>Analyzes competition, finance, risk, marketing, and strategy.</span>
                    </div>

                    <div class="mini-card">
                        <b>03 · Human Review Loop</b><br>
                        <span>If the plan is not approved, FounderAI revises it automatically using your feedback.</span>
                    </div>
                </div>
                ''')

    with gr.Group(visible=False) as plan_page:
        gr.HTML("""
        <div class="hero">
            <div class="badge">FounderAI • Final Business Plan</div>
            <h1 class="hero-title">Your AI-generated business plan is ready.</h1>
            <p class="hero-text">
                Review the plan, approve it, or request improvements. If rejected, FounderAI will revise it automatically.
            </p>
        </div>
        """)

        plan_message = gr.HTML()

        with gr.Group(elem_classes="plan-display"):
            gr.Markdown("## Final Business Plan")
            final_plan_box = gr.Markdown(value="")

        with gr.Group(elem_classes="review-card"):
            gr.Markdown("## Review Decision")

            approval_input = gr.Radio(
                choices=["Approved", "Not Approved"],
                label="Approval",
                value="Approved"
            )

            additional_info = gr.Textbox(
                label="Additional Information",
                placeholder="If Not Approved, write what needs to be improved...",
                lines=4
            )

            with gr.Row():
                review_button = gr.Button("Submit Review", variant="primary")
                back_button = gr.Button("Back to Startup Form")

    start_button.click(
        fn=start_and_show_plan,
        inputs=[
            project_name,
            city,
            budget,
            business_type,
            description
        ],
        outputs=[
            form_page,
            plan_page,
            form_message,
            plan_message,
            final_plan_box,
            additional_info,
            start_button
        ],
        show_progress="full"
    )

    review_button.click(
        fn=review_plan,
        inputs=[
            approval_input,
            additional_info
        ],
        outputs=[
            plan_message,
            final_plan_box,
            additional_info
        ]
    )

    back_button.click(
        fn=back_to_form,
        inputs=[],
        outputs=[
            form_page,
            plan_page
        ]
    )

website_demo.queue()
website_demo.launch(share=True, show_error=True)

Closing server running on port: 7860


/tmp/ipykernel_2224/448988665.py:369: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=website_css, theme=gr.themes.Soft()) as website_demo:
/tmp/ipykernel_2224/448988665.py:369: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=website_css, theme=gr.themes.Soft()) as website_demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02a2f7b462a74d332b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
